In [ ]:
# PARAMETERS — edit this cell only.
from __future__ import annotations

import json
import os
import re
import sys
from pathlib import Path


def locate_bundle() -> Path:
    starts = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for candidate in starts:
        if (candidate / "data" / "example_tiles.csv").is_file():
            return candidate
        nested = candidate / "tasks" / "260731_2_mllm_notebook_bundle"
        if (nested / "data" / "example_tiles.csv").is_file():
            return nested
    return Path.cwd().resolve()


BUNDLE_DIR = locate_bundle()
PROJECT_DIR = next(
    (candidate for candidate in [BUNDLE_DIR, *BUNDLE_DIR.parents]
     if (candidate / "experiments").is_dir() and (candidate / "scripts").is_dir()),
    BUNDLE_DIR,
)
sys.path.append(str(PROJECT_DIR))

# Required for live tile loading. Leave empty for the included six-tile examples.
COUNTING_DATA_ROOT = os.environ.get("COUNTING_DATA_ROOT", "")
TASKS = [
    "malaria/trophozoite", "malaria/difficult", "malaria/red_blood_cell",
    "birds/bird", "dota/tennis_court", "dota/harbor", "dota/ship",
    "dota/plane", "dota/small_vehicle", "kenyan_wildlife/giraffe",
    "dota/helicopter", "dota/storage_tank",
]
MODEL_LABEL = "Qwen3-VL-8B-Instruct"
BATCH_SIZE = 8
MAX_CHOICE = 300
GPU_MEMORY_UTILIZATION = 0.85
LABEL_ROOT = PROJECT_DIR / "outputs" / "mllm_all_tiles_all_models"
PROPRIETARY_ROOT = PROJECT_DIR / "outputs" / "proprietary_all_tiles_260710"
RUN_ROOT = PROJECT_DIR / "outputs" / "runs"
EXAMPLE_CSV = BUNDLE_DIR / "data" / "example_tiles.csv"
OPEN_SOURCE_CSV = PROJECT_DIR / "outputs" / "runs" / "open_source_all_tiles_current.csv" if (PROJECT_DIR / "outputs" / "runs" / "open_source_all_tiles_current.csv").is_file() else BUNDLE_DIR / "data" / "open_source_reference_sample.csv"
PROPRIETARY_CSV = PROJECT_DIR / "outputs" / "runs" / "proprietary_all_tiles_current.csv" if (PROJECT_DIR / "outputs" / "runs" / "proprietary_all_tiles_current.csv").is_file() else BUNDLE_DIR / "data" / "proprietary_reference_sample.csv"
OPENROUTER_KEY_SELECTOR = "OPENROUTER_API_KEY"

try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except NameError:
    pass

# Safety default: prepare and inspect only. Set True for paid API requests.
RUN_REQUESTS = False
MAX_REQUESTS = 12
WORKERS = 4
REQUEST_TIMEOUT_SEC = 900


# 02 — Proprietary API inference with staged caching

The original proprietary workflow has three meaningful stages: materialize the image/request, collect the natural-language count, then collect the integer-choice answer. This notebook keeps that shape in a small JSONL cache and uses a bounded local thread pool. The only external dependency is network access to OpenRouter.

The default is a dry preview. Set `RUN_REQUESTS=True` only after choosing a small `MAX_REQUESTS`, confirming the model name, and making the key selector available. The selector is `OPENROUTER_API_KEY`; the secret is resolved at runtime and never displayed or written.


In [ ]:
import base64
import concurrent.futures
import time
from io import BytesIO

import pandas as pd
import requests
from PIL import Image

MODELS = {"GPT-5.2": "openai/gpt-5.2", "Gemini-3.1-Pro": "google/gemini-3.1-pro-preview", "Qwen3.7-Plus": "qwen/qwen3.7-plus"}
MODEL_LABELS = ["GPT-5.2"]
API_ENDPOINT = "https://openrouter.ai/api/v1/chat/completions"
API_OUTPUT = BUNDLE_DIR / "proprietary_notebook_results.jsonl"


def resolve_api_key(selector: str) -> str:
    if os.environ.get(selector):
        return os.environ[selector]
    config_path = Path.home() / ".rc_config" / "api_keys"
    if config_path.is_file():
        for line in config_path.read_text().splitlines():
            name, separator, value = line.partition("=")
            if separator and name.strip() == selector and value.strip():
                return value.strip().strip('"').strip("'")
    raise EnvironmentError(f"No API key found for selector {selector!r}. Set that environment variable or configure the local key selector.")


def prompt_for(name: str) -> str:
    return f"How many {name} are in this image? Think briefly from visible evidence, then finish with a line exactly like Final answer: <number>."


def encode_image(path: Path) -> str:
    image = Image.open(path).convert("RGB")
    buffer = BytesIO()
    image.save(buffer, format="JPEG")
    return "data:image/jpeg;base64," + base64.b64encode(buffer.getvalue()).decode("ascii")


def query_openrouter(messages: list[dict], modelstr: str, api_key: str) -> tuple[dict, float]:
    started = time.perf_counter()
    response = requests.post(
        API_ENDPOINT,
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json", "X-OpenRouter-Metadata": "enabled"},
        json={"model": modelstr, "messages": messages, "stream": False, "temperature": 0, "max_tokens": 512},
        timeout=REQUEST_TIMEOUT_SEC,
    )
    response.raise_for_status()
    body = response.json()
    if not body.get("choices") or not body["choices"][0]["message"].get("content"):
        raise RuntimeError("OpenRouter returned no text content")
    return body, time.perf_counter() - started


def load_included_examples() -> pd.DataFrame:
    examples = pd.read_csv(EXAMPLE_CSV)
    examples["image_path"] = examples["image_file"].map(lambda name: BUNDLE_DIR / "assets" / name)
    return examples


def build_request_rows() -> list[dict]:
    examples = load_included_examples()
    rows = []
    for row in examples.itertuples(index=False):
        class_key = f"{row.dataset}/{row.task}"
        if class_key not in TASKS:
            continue
        rows.append({"class_key": class_key, "tile_index": int(row.tileid), "gt_count": int(row.ground_truth_label), "object_name": row.task.replace("_", " "), "image_path": str(row.image_path)})
    return rows[:MAX_REQUESTS]


def generate_and_extract(row: dict, model_label: str, api_key: str) -> dict:
    modelstr = MODELS[model_label]
    messages = [{"role": "user", "content": [{"type": "image_url", "image_url": {"url": encode_image(Path(row["image_path"]))}}, {"type": "text", "text": prompt_for(row["object_name"])}]}]
    body, elapsed = query_openrouter(messages, modelstr, api_key)
    nlg = body["choices"][0]["message"]["content"].strip()
    choice_prompt = "What number was this response trying to say?\n\nResponse:\n" + nlg + "\n\nAnswer only with one integer from 0 to 300."
    choice_body, choice_elapsed = query_openrouter([{"role": "user", "content": choice_prompt}], modelstr, api_key)
    choice = choice_body["choices"][0]["message"]["content"].strip()
    match = re.search(r"\d+", choice)
    return {**row, "model_label": model_label, "modelstr": modelstr, "reasoning": nlg, "nlg2choice": choice, "answer": int(match.group(0)) if match else -1, "generation_elapsed_sec": round(elapsed, 3), "choice_elapsed_sec": round(choice_elapsed, 3), "usage": {"generation": body.get("usage", {}), "choice": choice_body.get("usage", {})}}


## Run the staged API calls

Each completed row is appended immediately. Re-running the notebook can therefore be resumed by filtering the same row key before making another paid call. The small included examples are the default input; replace `build_request_rows` with a manifest-backed tile loader for the full frozen pools.


In [ ]:
def append_result(row: dict) -> None:
    API_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    with API_OUTPUT.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, sort_keys=True) + "\n")


if RUN_REQUESTS:
    api_key = resolve_api_key(OPENROUTER_KEY_SELECTOR)
    request_rows = build_request_rows()
    with concurrent.futures.ThreadPoolExecutor(max_workers=WORKERS) as executor:
        futures = [executor.submit(generate_and_extract, row, label, api_key) for label in MODEL_LABELS for row in request_rows]
        for future in concurrent.futures.as_completed(futures):
            result = future.result()
            append_result(result)
            print({"class_key": result["class_key"], "tile_index": result["tile_index"], "model": result["model_label"], "answer": result["answer"]})
else:
    print("Preview mode: no paid requests made. Set RUN_REQUESTS=True in the parameter cell to call OpenRouter.")


## Visualize the API stage

The plot is useful even before a live request: it compares the included reference counts with the previously exported proprietary predictions. If this notebook has just written its own JSONL, it uses those rows instead.


In [ ]:
from mu.visv2.figure import Figure

if API_OUTPUT.is_file():
    api_rows = pd.read_json(API_OUTPUT, lines=True)
else:
    api_rows = pd.read_csv(PROPRIETARY_CSV)
    api_rows["class_key"] = api_rows["dataset"] + "/" + api_rows["task"]
    api_rows["answer"] = api_rows["final_value"]
api_rows["absolute_error"] = (api_rows["answer"] - api_rows["ground_truth_label"]).abs()
fig = Figure(n_rows=1, n_cols=2, figsize=(12, 4.5), fontfamily="sans-serif", fontsize=11, panel_padding=(0.28, 0.2))
left, right = fig.panels
left.p.scatter(api_rows["ground_truth_label"], api_rows["answer"], s=16, alpha=0.35, color="#7c3aed")
maximum = max(api_rows["ground_truth_label"].max(), api_rows["answer"].max())
left.p.plot([0, maximum], [0, maximum], "--", color="#dc2626")
left.set_xaxislabel("true count")
left.set_yaxislabel("API count")
left.set_title("API predictions versus truth")
right.p.hist(api_rows["absolute_error"], bins=30, color="#c2410c", alpha=0.82)
right.set_xaxislabel("absolute error")
right.set_yaxislabel("tiles")
right.set_title("Error distribution")
fig.show()


For a full-pool run, point the request builder at a manifest with image paths and keep the same staged schema. The key selector and response rows remain secret-free; do not add the key, bearer header, or data URLs to notebook output or committed cache.
